In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as sps
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor

rs = 123 #Fix random state for reproducability
block_size = 25
n_bootstraps = 1000

#Copy over winning model data for each river
verde_downstream_features = [
    "Temp_Verde",
    "Precip_Verde",
    "Temp_Maricopa",
    "Precip_Maricopa",
    "Population",
    "IrrigatedLand_Acres",
    "datacenters_TotalMW",
    "datacenters_TotalNum",
    "log_flow_verde_upstream",
    "log_flow_verde_downstream_lag_5",
    "log_flow_verde_downstream_lag_6"
]

verde_downstream_model = XGBRegressor
verde_downstream_hp = {'max_depth': 6, 'n_estimators': 100}

verde_upstream_features = [
    "Temp_Verde",
    "Precip_Verde",
    "log_flow_verde_upstream_lag_1",
    "log_flow_verde_upstream_lag_11"
]

verde_upstream_model = GradientBoostingRegressor
verde_upstream_hp = {'max_depth': 3, 'n_estimators': 100}

salt_downstream_features = [
    "Temp_Salt",
    "Precip_Salt",
    "Temp_Maricopa",
    "Precip_Maricopa",
    "Population",
    "IrrigatedLand_Acres",
    "datacenters_TotalMW",
    "datacenters_TotalNum",
    "log_flow_salt_upstream",
    "log_flow_salt_downstream_lag_9",
    "log_flow_salt_downstream_lag_12"
]

salt_downstream_model = RandomForestRegressor
salt_downstream_hp = {'max_depth': 6, 'n_estimators': 100}

salt_upstream_features = [
    "Temp_Salt",
    "Precip_Salt",
    "log_flow_salt_upstream_lag_1",
    "log_flow_salt_upstream_lag_12"
]

salt_upstream_model = RandomForestRegressor
salt_upstream_hp = {'max_depth': 10, 'n_estimators': 300}

df = pd.read_csv('../data/formatteddata/combined_monthly_data.csv')


In [5]:
'''
Generates n_bootstraps block bootstrap resamples of the time series data from df_model_train in blocks of block_size
Trains model with hyperparameters hps on features from bootstrapped features and predicts target on df_model_train features
Returns bootstrapped predictions: 
Fixes random state rs for reproducibility
'''

def block_bootstrap_resampling(df_model_train, features, target, model, hps, block_size, n_bootstraps, rs, selectstring):

    

    n_samples = len(df_model_train)
    n_blocks_needed = int(np.ceil(n_samples / block_size))

    # Array to hold predictions from all 100 bootstrap models
    # Rows = test samples, Columns = bootstrap iterations
    all_bootstrap_preds = np.zeros((n_samples, n_bootstraps))

    print(f"Bootstrapping {n_bootstraps} resamples for {selectstring}...")

    for i in range(n_bootstraps):
        # Fix random seed per iteration for reproducibility, but vary it across loops
        np.random.seed(rs + i)
    
        # Draw random block starting positions
        start_indices = np.random.randint(0, n_samples - block_size, size=n_blocks_needed)
    
        # Stitch blocks together to build the bootstrapped training set
        bootstrap_indices = []
        for start in start_indices:
            bootstrap_indices.extend(list(range(start, start + block_size)))
    
        # Clip to original length to match exact dataset size
        bootstrap_indices = bootstrap_indices[:n_samples]
        df_bootstrapped = df_model_train.iloc[bootstrap_indices]
    
        X_train_b = df_bootstrapped[features]
        y_train_b = df_bootstrapped[target]
    
        # Initialize your best performing model hyperparameters
        model_b = model(**hps, random_state=rs+i)
        model_b.fit(X_train_b, y_train_b)
    
        # Predict on the original continuous dataset to see prediction variations
        all_bootstrap_preds[:, i] = model_b.predict(df_model_train[features])

    print("Bootstrap complete!")
    return all_bootstrap_preds


In [6]:
for riverselect in ['salt', 'verde']:
    for updownselect in ['up', 'down']:
        
        selectstring = f'{str.lower(riverselect)}_{str.lower(updownselect)}stream'
        #Set variables for selected river and side
        features = globals()[selectstring+'_features']
        target = f'log_flow_{selectstring}'
        model = globals()[selectstring+'_model']
        hps = globals()[selectstring+'_hp']

        all_cols_needed = features + [target]
        df_train = df[all_cols_needed]
        df_model_train = df_train.dropna(subset=all_cols_needed).reset_index(drop=True)

        X_train_a = df_model_train[features]
        y_train_a = df_model_train[target]

        #Actual model predictions to compare bootstrapped predictions with
        model_a = model(**hps, random_state=rs)
        model_a.fit(X_train_a, y_train_a)
        model_preds = model_a.predict(df_model_train[features])

        all_bootstrap_preds = block_bootstrap_resampling(df_model_train, features, target, model, hps, block_size, n_bootstraps, rs, selectstring)
    
        std_predictions = np.std(all_bootstrap_preds, axis=1)
        mean_predictions = np.mean(all_bootstrap_preds, axis=1)
        z_score = (model_preds - mean_predictions)/std_predictions

        print(f'{selectstring} uncertainty: {np.quantile(std_predictions, q=0.75)}')


Bootstrapping 1000 resamples for salt_upstream...
Bootstrap complete!
salt_upstream uncertainty: 0.18156243536199843
Bootstrapping 1000 resamples for salt_downstream...
Bootstrap complete!
salt_downstream uncertainty: 0.4911335915388519
Bootstrapping 1000 resamples for verde_upstream...
Bootstrap complete!
verde_upstream uncertainty: 0.14159225368829015
Bootstrapping 1000 resamples for verde_downstream...
Bootstrap complete!
verde_downstream uncertainty: 0.5146019846927228
